# ConvNeXt 224 + 320: OOF diagnostics and fixed 50/50 blend

No GPU, training, new labels or additional models. Read the two complete CV outputs and
their ensemble probability CSVs. Average 224 and 320 **probabilities** with a fixed 50/50
weight, separately for original and TTA inference. Fit per-class thresholds only on the
matching OOF blend. No leaderboard-guided weight search and no averaging of binary labels.

Report per-class F1, precision, recall, FP/FN, and threshold cross-fitting: fit thresholds
on four OOF folds and score the fifth, then aggregate the held-out predictions. This
reduces direct threshold fitting on the scored labels, but checkpoint selection already
used those folds, so it is not a fully nested independent evaluation. Neither OOF report
is a guarantee of the public/private test score. Keep the 0.94275 baseline.

Inputs must be from the same images, folds and corresponding checkpoint manifests.
Rows are aligned by `image_id`. Both methods export probabilities and binary submissions
to a separate `convnext_blend_224_320` directory. This notebook never submits automatically.

In [1]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score
from IPython.display import display, FileLink

BASE_CV_DIR = Path('/kaggle/working/convnext_cv')
FINE_CV_DIR = Path('/kaggle/working/convnext_cv_320')
BASE_PREDICTIONS_DIR = Path('/kaggle/working/convnext_cv_ensemble')
FINE_PREDICTIONS_DIR = Path('/kaggle/working/convnext_cv_320_ensemble')
DATA_DIR = Path('/kaggle/input/competitions/cats-and-dogs')
OUTPUT_DIR = Path('/kaggle/working/convnext_blend_224_320')
LABEL_COLS = ['Cat', 'Dog', 'Moris', 'Motya', 'Biatrix']

## 1. Verify corresponding OOF and test probabilities

Set paths explicitly if the outputs are mounted under Kaggle Input. No split is generated.

In [2]:
base_manifest = json.loads((BASE_CV_DIR / 'ensemble_config.json').read_text())
fine_manifest = json.loads((FINE_CV_DIR / 'ensemble_config.json').read_text())
assert base_manifest['folds_fingerprint'] == fine_manifest['folds_fingerprint']
for manifest, size in [(base_manifest, 224), (fine_manifest, 320)]:
    assert manifest['model_name'] == 'convnext_tiny.fb_in1k' and manifest['label_cols'] == LABEL_COLS
    assert manifest['folds'] == list(range(5)) and manifest['preprocessing']['image_size'] == size
for directory, manifest in [(BASE_PREDICTIONS_DIR, base_manifest), (FINE_PREDICTIONS_DIR, fine_manifest)]:
    prediction_manifest = json.loads((directory / 'ensemble_config.json').read_text())
    assert prediction_manifest['cv_fingerprint'] == manifest['cv_fingerprint']
    assert prediction_manifest['checkpoint_hashes'] == manifest['checkpoint_hashes']

def read_unique(path):
    frame = pd.read_csv(path, dtype={'image_id': str})
    assert len(frame) > 0 and frame.image_id.notna().all() and frame.image_id.is_unique, path
    return frame

def align(frame, ids):
    assert set(frame.image_id) == set(ids), 'Image IDs differ.'
    return frame.set_index('image_id').loc[list(ids)].reset_index()

def probabilities(frame, columns):
    values = frame[columns].to_numpy(dtype=np.float64)
    assert values.shape == (len(frame), len(LABEL_COLS)) and np.isfinite(values).all()
    assert ((values >= 0) & (values <= 1)).all()
    return values

base_oof = read_unique(BASE_CV_DIR / 'oof_predictions.csv')
fine_oof = align(read_unique(FINE_CV_DIR / 'oof_predictions.csv'), base_oof.image_id)
assert np.array_equal(base_oof[LABEL_COLS].to_numpy(), fine_oof[LABEL_COLS].to_numpy())
assert np.array_equal(base_oof[['fold', 'cluster']].to_numpy(), fine_oof[['fold', 'cluster']].to_numpy())
assert set(base_oof.fold.unique()) == set(range(5))
assert base_oof.groupby('cluster').fold.nunique().eq(1).all()
assert base_oof[LABEL_COLS].isin([0, 1]).all().all()
targets = base_oof[LABEL_COLS].to_numpy(dtype=np.int32)
fold_ids = base_oof.fold.to_numpy()
print('Aligned OOF rows:', len(base_oof))

Aligned OOF rows: 12260


In [3]:
THRESHOLD = 0.5

def tune_thresholds(targets, probabilities):
    thresholds = np.full(len(LABEL_COLS), THRESHOLD, dtype=np.float64)
    candidates = np.round(np.linspace(0.05, 0.95, 91), 2)
    for j, label in enumerate(LABEL_COLS):
        if np.unique(targets[:, j]).size < 2:
            print(f'{label}: keeping {THRESHOLD}; validation does not contain both outcomes.')
            continue
        scores = np.array([
            f1_score(targets[:, j], probabilities[:, j] >= t, zero_division=0)
            for t in candidates
        ])
        tied = np.flatnonzero(np.isclose(scores, scores.max(), rtol=0, atol=1e-12))
        best = tied[np.argmin(np.abs(candidates[tied] - THRESHOLD))]
        thresholds[j] = candidates[best]
    return thresholds

## 2. Compare the two resolutions and a predefined equal blend

Original and TTA variants are evaluated separately. Threshold cross-fitting never tunes on the fold being scored. Full-OOF fitted thresholds are used only for final test export.

In [4]:
candidate_probabilities, fitted_thresholds, comparison_rows, per_label_rows = {}, {}, [], []
for method, suffix in [('original', '_prob'), ('tta', '_prob_tta')]:
    p_base = probabilities(base_oof, [label + suffix for label in LABEL_COLS])
    p_fine = probabilities(fine_oof, [label + suffix for label in LABEL_COLS])
    for name, values in [('224', p_base), ('320', p_fine), ('blend', (p_base + p_fine) / 2)]:
        key = f'{name}_{method}'
        candidate_probabilities[key] = values
        cutoffs = tune_thresholds(targets, values)
        fitted_thresholds[key] = cutoffs
        cross_fitted = np.empty_like(targets)
        for fold in range(5):
            validation = fold_ids == fold
            calibration = ~validation
            fold_cutoffs = tune_thresholds(targets[calibration], values[calibration])
            cross_fitted[validation] = (values[validation] >= fold_cutoffs).astype(np.int32)
        fitted = values >= cutoffs
        comparison_rows.append({'candidate': key,
            'f1_at_0_5': f1_score(targets, values >= 0.5, average='macro', zero_division=0),
            'f1_fitted_oof': f1_score(targets, fitted, average='macro', zero_division=0),
            'f1_cross_fitted_thresholds': f1_score(targets, cross_fitted, average='macro', zero_division=0)})
        for j, label in enumerate(LABEL_COLS):
            y, predicted = targets[:, j], cross_fitted[:, j]
            per_label_rows.append({'candidate': key, 'label': label, 'positives': int(y.sum()),
                'f1_cross_fitted': f1_score(y, predicted, zero_division=0),
                'precision_cross_fitted': precision_score(y, predicted, zero_division=0),
                'recall_cross_fitted': recall_score(y, predicted, zero_division=0),
                'false_positives': int(((predicted == 1) & (y == 0)).sum()),
                'false_negatives': int(((predicted == 0) & (y == 1)).sum()), 'final_threshold': float(cutoffs[j])})
comparison = pd.DataFrame(comparison_rows).sort_values('f1_cross_fitted_thresholds', ascending=False)
per_label = pd.DataFrame(per_label_rows)
display(comparison)
display(per_label)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
comparison.to_csv(OUTPUT_DIR / 'oof_comparison.csv', index=False)
per_label.to_csv(OUTPUT_DIR / 'oof_per_label.csv', index=False)

,candidate,f1_at_0_5,f1_fitted_oof,f1_cross_fitted_thresholds
2,blend_original,0.977012,0.981095,0.976955
0,224_original,0.976454,0.978271,0.976454
1,320_original,0.976614,0.978687,0.970514
5,blend_tta,0.974745,0.978988,0.970500
4,320_tta,0.974270,0.978614,0.968212
3,224_tta,0.969678,0.974188,0.961489


,candidate,label,positives,f1_cross_fitted,precision_cross_fitted,recall_cross_fitted,false_positives,false_negatives,final_threshold
0,224_original,Cat,5077,0.994777,0.995464,0.994091,23,30,0.39
1,224_original,Dog,5025,0.992834,0.993032,0.992637,35,37,0.39
2,224_original,Moris,56,0.954955,0.963636,0.946429,2,3,0.62
3,224_original,Motya,42,0.964706,0.953488,0.976190,2,1,0.50
4,224_original,Biatrix,41,0.975000,1.000000,0.951220,0,2,0.50
5,320_original,Cat,5077,0.994685,0.994098,0.995273,30,24,0.34
6,320_original,Dog,5025,0.993929,0.994225,0.993632,29,32,0.42
7,320_original,Moris,56,0.946429,0.946429,0.946429,3,3,0.50
8,320_original,Motya,42,0.942529,0.911111,0.976190,4,1,0.66
9,320_original,Biatrix,41,0.975000,1.000000,0.951220,0,2,0.50


## 3. Export equal-blend submissions

Requires the original and TTA test probability CSVs from both ensemble notebooks. Select the matching official competition data path. The baseline files are never overwritten.

In [5]:
sample = read_unique(DATA_DIR / 'sample_submission.csv')
test_ids_frame = read_unique(DATA_DIR / 'test.csv')
assert set(sample.columns) == {'image_id', *LABEL_COLS}
assert set(sample.image_id) == set(test_ids_frame.image_id)
export_metadata = {'weights': {'224': 0.5, '320': 0.5}, 'label_cols': LABEL_COLS,
                   'base_cv_fingerprint': base_manifest['cv_fingerprint'],
                   'fine_cv_fingerprint': fine_manifest['cv_fingerprint'], 'thresholds': {}}
for method, filename in [('original', 'submission_blend_224_320.csv'), ('tta', 'submission_blend_224_320_tta.csv')]:
    base_test = align(read_unique(BASE_PREDICTIONS_DIR / f'test_probabilities_{method}.csv'), sample.image_id)
    fine_test = align(read_unique(FINE_PREDICTIONS_DIR / f'test_probabilities_{method}.csv'), sample.image_id)
    blended = (probabilities(base_test, LABEL_COLS) + probabilities(fine_test, LABEL_COLS)) / 2
    probability_frame = pd.DataFrame(blended, columns=LABEL_COLS)
    probability_frame.insert(0, 'image_id', sample.image_id.to_numpy())
    probability_frame.to_csv(OUTPUT_DIR / f'test_probabilities_blend_{method}.csv', index=False)
    cutoffs = fitted_thresholds[f'blend_{method}']
    submission = probability_frame.copy()
    submission[LABEL_COLS] = (blended >= cutoffs).astype(np.int32)
    assert len(submission) == len(sample) and submission[LABEL_COLS].isin([0, 1]).all().all()
    path = OUTPUT_DIR / filename
    submission.to_csv(path, index=False)
    export_metadata['thresholds'][method] = cutoffs.tolist()
    print('Saved:', path, '| rows:', len(submission))
    display(FileLink(str(path)))
(OUTPUT_DIR / 'blend_config.json').write_text(json.dumps(export_metadata, indent=2))

Saved: /kaggle/working/convnext_blend_224_320/submission_blend_224_320.csv | rows: 18673


/kaggle/working/convnext_blend_224_320/submission_blend_224_320.csv

Saved: /kaggle/working/convnext_blend_224_320/submission_blend_224_320_tta.csv | rows: 18673


/kaggle/working/convnext_blend_224_320/submission_blend_224_320_tta.csv

507